# 4.1 Primary outcome - simple diff in diff analysis

This notebook does the following:
    - OLS analysis of secondary outcomes - climate attitudes and top-down vs bottom-up attitude

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
import importlib
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
import os
import statsmodels.formula.api as smf
import pyfixest as pf
from make_regression_table import make_regression_table

In [2]:
# Load data
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv", 
    keep_default_na=False, # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [3]:
# Construct post variable
df["post"] = (df["survey"] == "EL").astype(int)

In [4]:
# Keep only labgroups that have both pre and post observations
labgroup_counts = df.groupby("labgroupid")["survey"].nunique()
labgroups_to_keep = labgroup_counts[labgroup_counts == 2].index
df = df[df["labgroupid"].isin(labgroups_to_keep)].copy()

## (1) Prepare dataset

In [5]:
# Create baseline_energy_use variable
df["baseline_energy_use"] = df.groupby("labgroupid")["annual_electricity_total"].transform(
    lambda x: x[df["survey"] == "BL"].iloc[0]
)

# Log transform the baseline_energy_use variable
df["log_baseline_energy_use"] = np.log(df["baseline_energy_use"])

In [6]:
# Keep only one row per labgroupid - we will use the EL observation
df = df[df["survey"] == "EL"].copy()

In [7]:
print(df.groupby("faculty")["labgroupid"].nunique())

faculty
Both MNF and MeF              5
Faculty of Medicine (MeF)    27
Faculty of Science (MNF)     77
Name: labgroupid, dtype: int64


In [8]:
# Create mnf_indicator variable
df["mnf_indicator"] = df["faculty"].apply(lambda x: 1 if x == "Faculty of Science (MNF)" else 0).astype(int)

In [9]:
# Recode attitude responses to numeric scale
attitude_map = {
    "Strongly disagree": 1,
    "Disagree": 2,
    "Neither agree nor disagree": 3,
    "Agree": 4,
    "Strongly agree": 5,
}

# Keep attitude variables in numerical order
attitude_cols = sorted(
    [c for c in df.columns if c.startswith("attitude_q_")],
    key=lambda x: int(re.search(r"(\d+)$", x).group(1))
)

In [10]:
# Check whether any labs that answer one attitude q but have missing values for another attitude q
for att_q in attitude_cols:
    missing_att_q = df[df[att_q].isna()]["labgroupid"]
    for other_att_q in attitude_cols:
        if other_att_q != att_q:
            missing_other_att_q = df[df[other_att_q].isna()]["labgroupid"]
            if not missing_att_q.equals(missing_other_att_q):
                print(f"Warning: Labs with missing {att_q} do not match labs with missing {other_att_q}")

In [11]:
# Restrict to labs with non-missing values for all attitude questions
df = df.dropna(subset=attitude_cols).copy()

## (2) Regressions

### (2a) No enumerator FEs

In [12]:
# Define regression formula

# Control vars
control_vars = [
    "no_researchers",
    "log_baseline_energy_use",
    "mnf_indicator",
]

# Independent vars
ind_vars = "treated + " + " + ".join(control_vars)
print(ind_vars)

treated + no_researchers + log_baseline_energy_use + mnf_indicator


In [13]:
# Regressions without enumerator FEs
fit_attitudes = {}

for att_q in attitude_cols:

    # Map attitude responses to numeric scale for regression
    df["y"] = df[att_q].map(attitude_map)

    formula = f"y ~ {ind_vars}"
    model = smf.ols(formula=formula, data=df)
    fit_attitudes[att_q] = model.fit(cov_type="cluster", cov_kwds={"groups": df["enum_id"]})
    print(f"Finished regression for {att_q}")
    print(fit_attitudes[att_q].summary())

Finished regression for attitude_q_1
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.033
Model:                            OLS   Adj. R-squared:                 -0.011
Method:                 Least Squares   F-statistic:                    0.9994
Date:                Wed, 10 Jun 2026   Prob (F-statistic):              0.424
Time:                        17:20:37   Log-Likelihood:                -104.00
No. Observations:                  93   AIC:                             218.0
Df Residuals:                      88   BIC:                             230.7
Df Model:                           4                                         
Covariance Type:              cluster                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------

### (2b) With enumerator FEs

In [14]:
# Restrict data to enums that are non-singleton and have both treated and control labs
all_enums = df["enum_id"].unique()

enum_counts = df["enum_id"].value_counts()
singleton_enums = enum_counts[enum_counts == 1].index

enum_t_labs = df[df["treated"] == 1]["enum_id"].unique()
enum_c_labs  = df[df["treated"] == 0]["enum_id"].unique()

# Enums that only appear in treated or only in control labs
no_t_enums = all_enums[~np.isin(all_enums, enum_t_labs)]
no_c_enums = all_enums[~np.isin(all_enums, enum_c_labs)]

enums_to_exclude = set(singleton_enums) | set(no_t_enums) | set(no_c_enums)
enum_fe_df = df[~df["enum_id"].isin(enums_to_exclude)].copy()

print(f"Enums excluded: {len(enums_to_exclude)}. This comprises: singletons: {len(singleton_enums)}, T-only: {len(no_t_enums)}, C-only: {len(no_c_enums)}")
print(f"Rows remaining: {len(enum_fe_df)} of {len(df)}")

Enums excluded: 8. This comprises: singletons: 6, T-only: 2, C-only: 6
Rows remaining: 82 of 93


In [15]:
# Regressions with enumerator FEs
fit_attitudes_enum_fe = {}

for att_q in attitude_cols:

    # Map attitude responses to numeric scale for regression
    enum_fe_df["y"] = enum_fe_df[att_q].map(attitude_map)

    fit_attitudes_enum_fe[att_q] = pf.feols(
                "y ~ treated + no_researchers + mnf_indicator + log_baseline_energy_use | enum_id",
                data=enum_fe_df,
                vcov={"CRV1": "enum_id"}
            )
    print(fit_attitudes_enum_fe[att_q].summary())

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


###

Estimation:  OLS
Dep. var.: y, Fixed effects: enum_id
sample: None = all
Inference:  CRV1
Observations:  82

| Coefficient             |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| treated                 |     -0.095 |        0.132 |    -0.718 |      0.481 | -0.370 |   0.180 |
| no_researchers          |      0.002 |        0.012 |     0.137 |      0.892 | -0.023 |   0.026 |
| mnf_indicator           |      0.218 |        0.224 |     0.976 |      0.341 | -0.248 |   0.685 |
| log_baseline_energy_use |     -0.007 |        0.077 |    -0.094 |      0.926 | -0.169 |   0.154 |
---
RMSE: 0.625 R2: 0.347 R2 Within: 0.023 
None
###

Estimation:  OLS
Dep. var.: y, Fixed effects: enum_id
sample: None = all
Inference:  CRV1
Observations:  82

| Coefficient             |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:---------------------

## (3) Make regression tables

In [16]:
# Column names
col_subgroups = {
        "University responsibility":         [0],
        "Excess energy use": [1],
        "Try consider sustainability":        [2],
        "Slow down research":       [3],
        "Improve research quality":           [4],
        "Awareness of energy costs":      [5],
        "Financial motivation":           [6],
        "Top-down more effective":     [7],
    }

In [17]:
# Create regression table without enumerator FEs
table = make_regression_table(
    fit_list    = [fit_attitudes[q] for q in attitude_cols],
    model_names = [f"({i+1})" for i in range(len(attitude_cols))],
    keep_vars   = ["treated"],
    var_labels  = {"treated": "Treated"},
    col_subgroups = col_subgroups,
    fe_rows     = None,
    baseline_mean = None,
    decimals      = 3,
    r2_type       = None,
    col1_width    = "4cm",
    coln_width    = "2cm",
)
table_path = config.OUTPUT / "5_Regression_Tables" / "secondary_attitudes_no_enum_fe.tex"
_ = table_path.write_text(table)

In [18]:
# Create regression table with enumerator FEs
table = make_regression_table(
    fit_list    = [fit_attitudes_enum_fe[q] for q in attitude_cols],
    model_names = [f"({i+1})" for i in range(len(attitude_cols))],
    keep_vars   = ["treated"],
    var_labels  = {"treated": "Treated"},
    col_subgroups = col_subgroups,
    fe_rows     = {
        "Enumerator FE": [True] * len(attitude_cols),
    },
    baseline_mean = None,
    decimals      = 3,
    r2_type       = None,
    col1_width    = "4cm",
    coln_width    = "2cm",
)
table_path = config.OUTPUT / "5_Regression_Tables" / "secondary_attitudes_enum_fe.tex"
_ = table_path.write_text(table)